<a href="https://colab.research.google.com/github/AngeNana23/Assignment-13/blob/main/Assignment13.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

https://colab.research.google.com/drive/1Kjcs1GSTFkBt6yDmeBx787VzVwN61ObD?usp=sharing



In [ ]:
# import libraries
import numpy as np
import tensorflow as tf
import requests
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# 1. DATASET (Project Gutenberg)

url = "https://www.gutenberg.org/files/11/11-0.txt"  # Alice in Wonderland
text = requests.get(url).text.lower()

# Reduce size for stability (prevents crashes in Colab)
text = text[:100000]

In [ ]:
tokenizer = Tokenizer(num_words=5000, oov_token="<OOV>")
tokenizer.fit_on_texts([text])

total_words = len(tokenizer.word_index) + 1

words = text.split()

# limit sequence creation
MAX_WORDS = 3000

input_sequences = []

for i in range(1, min(len(words), MAX_WORDS)):
    seq = words[:i+1]
    token_list = tokenizer.texts_to_sequences([' '.join(seq)])[0]

    if len(token_list) > 0:
        input_sequences.append(token_list)

# limit sequence length
max_len = min(max(len(x) for x in input_sequences), 40)

input_sequences = pad_sequences(
    input_sequences,
    maxlen=max_len,
    padding='pre'
)

# Inputs / Labels
X = input_sequences[:, :-1]
y = input_sequences[:, -1]

# extra safety limit
X = X[:15000]
y = y[:15000]

In [ ]:
# 3. GPT-STYLE TRANSFORMER MODEL

embed_dim = 64
seq_len = X.shape[1]

inputs = tf.keras.Input(shape=(seq_len,))

# Embedding
x = tf.keras.layers.Embedding(total_words, embed_dim)(inputs)

# Multi-head Attention (core GPT concept)
attn = tf.keras.layers.MultiHeadAttention(
    num_heads=2,
    key_dim=embed_dim
)(x, x)

# Residual connection + normalization
x = tf.keras.layers.Add()([x, attn])
x = tf.keras.layers.LayerNormalization()(x)

# Feedforward network
ff = tf.keras.layers.Dense(embed_dim * 2, activation='relu')(x)
ff = tf.keras.layers.Dense(embed_dim)(ff)

x = tf.keras.layers.Add()([x, ff])
x = tf.keras.layers.LayerNormalization()(x)

# Pooling
x = tf.keras.layers.GlobalAveragePooling1D()(x)

# Output layer
outputs = tf.keras.layers.Dense(total_words, activation='softmax')(x)

model = tf.keras.Model(inputs, outputs)

In [ ]:
# 4. COMPILATION

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 39)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, 39, 64)    │    159,168 │ input_layer_1[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 39, 64)    │     33,216 │ embedding_1[0][0… │
│ (MultiHeadAttentio… │                   │            │ embedding_1[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_2 (Add)         │ (None, 39, 64)    │          0 │ embedding_1[0][0… │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 39, 64)    │        128 │ add_2[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 39, 128)   │      8,320 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 39, 64)    │      8,256 │ dense_3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_3 (Add)         │ (None, 39, 64)    │          0 │ layer_normalizat… │
│                     │                   │            │ dense_4[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 39, 64)    │        128 │ add_3[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 64)        │          0 │ layer_normalizat… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 2487)      │    161,655 │ global_average_p… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 370,871 (1.41 MB)

 Trainable params: 370,871 (1.41 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# 5. Training to preveint crash

history = model.fit(
    X,
    y,
    epochs=5,
    batch_size=32,
    verbose=1
)

Epoch 1/5
94/94 ━━━━━━━━━━━━━━━━━━━━ 7s 37ms/step - accuracy: 0.0403 - loss: 6.6014
Epoch 2/5
94/94 ━━━━━━━━━━━━━━━━━━━━ 9s 98ms/step - accuracy: 0.0467 - loss: 5.8188
Epoch 3/5
94/94 ━━━━━━━━━━━━━━━━━━━━ 5s 48ms/step - accuracy: 0.0507 - loss: 5.7179
Epoch 4/5
94/94 ━━━━━━━━━━━━━━━━━━━━ 6s 65ms/step - accuracy: 0.0610 - loss: 5.5876
Epoch 5/5
94/94 ━━━━━━━━━━━━━━━━━━━━ 4s 39ms/step - accuracy: 0.0674 - loss: 5.4200


In [ ]:
# 6. TEXT GENERATION

def sample(preds, temperature=0.8):
    preds = np.asarray(preds).astype("float64")
    preds = np.log(preds + 1e-8) / temperature
    exp_preds = np.exp(preds)
    preds = exp_preds / np.sum(exp_preds)
    return np.random.choice(len(preds), p=preds)

def generate_text(seed_text, next_words=25, temperature=0.8):

    for _ in range(next_words):

        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], maxlen=seq_len, padding='pre')

        preds = model.predict(token_list, verbose=0)[0]
        predicted_id = sample(preds, temperature)

        for word, index in tokenizer.word_index.items():
            if index == predicted_id:
                seed_text += " " + word
                break

    return seed_text

In [ ]:
# 7. APPLICATION DEMONSTRATION (CONTENT CREATOR TOOL)

def content_creation_app(prompt):
    print("\n===================================")
    print(" GENERATIVE AI CONTENT CREATOR ")
    print("===================================")
    print("Input Prompt:", prompt)
    print("\nGenerated Content:\n")

    result = generate_text(prompt, next_words=40, temperature=0.8)
    print(result)

    return result


In [ ]:
# 8. TEST THE MODEL

content_creation_app("the future of artificial intelligence")


 GENERATIVE AI CONTENT CREATOR 
Input Prompt: the future of artificial intelligence

Generated Content:

the future of artificial intelligence the chapter delight kind chapter party chapter puzzling the “oh all feet moment way struck curtseying chapter the was be chapter the opened gutenberg ever next on alice the they get the is likely the grow be the off the


'the future of artificial intelligence the chapter delight kind chapter party chapter puzzling the “oh all feet moment way struck curtseying chapter the was be chapter the opened gutenberg ever next on alice the they get the is likely the grow be the off the'